In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task='text-generation'
)

In [4]:
model = ChatHuggingFace(llm = llm)

In [5]:
# create state

class LLMState(TypedDict):

    question: str
    answer: str

In [6]:
def llm_QA(state: LLMState) -> LLMState:

    # extract the question
    question = state['question']

    # create a prompt template
    template = PromptTemplate(
        template='Give the answer to the following question \n {question}',
        input_variables=['question'],
    )

    # create the actual prompt
    prompt = template.invoke({"question": question})

    # ask that question to the LLM
    llm_answer = model.invoke(prompt)

    # update the answer in the state
    state['answer'] = llm_answer.content

    # return state
    return state

In [7]:
# create graph
graph = StateGraph(LLMState)

# add nodes to the graph
graph.add_node('llm_QA', llm_QA)

# add edges to the graph
graph.add_edge(START, 'llm_QA')
graph.add_edge('llm_QA', END)

# compile the graph
workflow = graph.compile()

In [8]:
# execute the graph
initial_state = {'question': 'Who is the Prime Minister of India?'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'Who is the Prime Minister of India?', 'answer': 'As of my last update in 2021, the Prime Minister of India is Narendra Modi. He has been in office since May 2014.'}
